In [ ]:
# LLM API 설정
LLM_API_URL = 'http://192.168.0.45:1234/v1/chat/completions'  # 필요시 변경
LLM_MODEL = 'google/gemma-4-31b'  # 필요시 변경

import pandas as pd
import numpy as np
import re
import csv
import time
import os
import requests
from tqdm import tqdm

PROTECTED_TOKEN_PATTERN = re.compile(r'<[^>]+>|\[\d+\]')

def split_edge_markers(text):
    if not isinstance(text, str) or not text:
        return '', text, ''

    prefix_match = re.match(r'^[^\w\[<]+\s*', text)
    suffix_match = re.search(r'\s*[^\w\]>]+$', text)

    prefix = prefix_match.group(0) if prefix_match else ''
    suffix = suffix_match.group(0) if suffix_match else ''

    start = len(prefix)
    end = len(text) - len(suffix) if suffix else len(text)
    core_text = text[start:end]
    return prefix, core_text, suffix

def protect_inline_tokens(text):
    protected_values = []

    def replacer(match):
        token_index = len(protected_values)
        protected_values.append(match.group(0))
        return f'__PROTECTED_{token_index}__'

    protected_text = PROTECTED_TOKEN_PATTERN.sub(replacer, text)
    return protected_text, protected_values

def restore_inline_tokens(text, protected_values):
    restored_text = text
    for token_index, original_value in enumerate(protected_values):
        restored_text = restored_text.replace(f'__PROTECTED_{token_index}__', original_value)
    return restored_text

def preserve_edge_markers(source_text, translated_text):
    if not isinstance(source_text, str) or not isinstance(translated_text, str):
        return translated_text

    prefix, _, suffix = split_edge_markers(source_text)
    result_text = translated_text.strip()
    if not result_text:
        return translated_text

    if prefix.strip() and not result_text.startswith(prefix.strip()):
        result_text = prefix + result_text.lstrip()

    if suffix.strip() and not result_text.endswith(suffix.strip()):
        result_text = result_text.rstrip() + suffix

    return result_text

def restore_identifier_column(source_df, target_df):
    if 'ID' in source_df.columns and 'ID' in target_df.columns:
        source_id = source_df['ID']
        if pd.api.types.is_numeric_dtype(source_id):
            target_df['ID'] = pd.to_numeric(source_id, errors='coerce').astype('Int64')
        else:
            target_df['ID'] = source_id

def LoadLLM(txt, extxt, src_lang, tgt_lang):
    if not txt or txt == extxt:
        return extxt

    prefix, core_text, suffix = split_edge_markers(txt)
    if not core_text.strip():
        return txt

    protected_core_text, protected_values = protect_inline_tokens(core_text)

    url = LLM_API_URL
    headers = {'Content-Type': 'application/json'}
    prompt = (
        f'Translate from {src_lang} to {tgt_lang}. '
        f'Return only the translated core text. '
        f'Keep tokens like __PROTECTED_0__ exactly unchanged. '
        f'Do not include leading prefix {prefix!r} or trailing suffix {suffix!r}; they are restored separately.\n\n'
        f'Core text: {protected_core_text}'
    )
    data = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    try:
        response = requests.post(url, headers=headers, json=data, timeout=60*3)  # 3분 타임아웃
        response.raise_for_status()
        result = response.json()
        translated_text = result['choices'][0]['message']['content'].strip()
        translated_text = restore_inline_tokens(translated_text, protected_values)
        return preserve_edge_markers(txt, translated_text)
    except Exception as e:
        print(f"번역 오류: {e}")
        return txt

def createFolder(directory):
    try:
        if not os.path.exists(directory):
            os.makedirs(directory)
    except OSError:
        print ('Error: Creating directory. ' +  directory)

def Convert(loadList, language, languageFull, replaceList, src_lang, tgt_lang, currentVersion):
    # 체크리스트 불러오기
    if os.path.isfile('./checkList.csv'):
        checkList = pd.read_csv('./checkList.csv', encoding = 'utf-8')
    else:
        checkList = pd.DataFrame(columns=['Language', 'File'])

    for loadFile in loadList:
        checkLanguage = False
        # 체크리스트
        for i in range(len(checkList)):
            if  checkList['Language'][i] == languageFull:
                if checkList['File'][i] == loadFile:
                    if str(checkList['Version'][i]) == currentVersion:
                        print(languageFull + ' ' + loadFile)
                        checkLanguage = True
                        break

        if checkLanguage == False:
            #데이터 불러오기
            originRead = pd.read_csv('./English/' + loadFile + '.csv', encoding = 'utf-8')
            current_read = originRead.copy()

            # 바꿀 데이터인지 확인
            isReplace = False
            for replaceData in replaceList:
                if loadFile == replaceData:
                    isReplace = True
                    break

            # 파일이 있어야 비교
            if os.path.isfile('./BeforeEnglish/' + loadFile + '.csv') and isReplace == False:
                before_read = pd.read_csv('./BeforeEnglish/' + loadFile + '.csv', encoding = 'utf-8')
                compare_columns = [
                    col for col in originRead.columns
                    if col in before_read.columns and any(key in col for key in ['Name', 'Dec'])
                ]
                compare_length = min(len(originRead), len(before_read))
                changed_mask = pd.Series(False, index=originRead.index)
                result = originRead.iloc[0:0].copy()

                if compare_length > 0 and compare_columns:
                    current_compare = originRead.iloc[:compare_length][compare_columns].fillna('')
                    before_compare = before_read.iloc[:compare_length][compare_columns].fillna('')
                    changed_mask.iloc[:compare_length] = current_compare.ne(before_compare).any(axis=1).to_numpy()

                if len(originRead) > len(before_read):
                    changed_mask.iloc[compare_length:] = True

                result = originRead.loc[changed_mask].copy()

                if not result.empty:
                    colList = ['Name', 'Dec']
                    exTxt = ''
                    exEx = ''
                    for col in colList:
                        for dfCol in result.columns:
                            if col in dfCol:
                                for r in result.index:
                                    source_text = result.at[r, col]
                                    if exEx == source_text:
                                        result.at[r, col] = exTxt
                                    else:
                                        van = LoadLLM(source_text, exTxt, src_lang, tgt_lang)
                                        exEx = source_text
                                        result.at[r, col] = van
                                        exTxt = van

                    languageRead = pd.read_csv('./'+ languageFull +'/' + loadFile + '.csv', encoding = 'utf-8')
                    languageRead = languageRead.reindex(originRead.index).copy()
                    setColList = ['Name', 'Dec']
                    for _index in result.index:
                        for col in setColList:
                            for dfCol in languageRead.columns:
                                if col in dfCol:
                                    languageRead.at[_index, dfCol] = result.at[_index, col]
                                    break
                    languageRead = languageRead.iloc[:len(originRead)]
                    restore_identifier_column(originRead, languageRead)
                    if loadFile == 'Etc':
                        languageRead['Korean'] = originRead['Korean']
                    createFolder('./' + languageFull)
                    languageRead.to_csv('./'+ languageFull +'/' + loadFile + '.csv', mode='w', index=False, encoding='utf-8-sig')
            else:
                colList = ['Name', 'Dec']
                exTxt = ''
                exEx = ''
                for col in colList:
                    for dfCol in current_read.columns:
                        if col in dfCol:
                            for r in tqdm(current_read.index):
                                if exEx == current_read.at[r, col]:
                                    current_read.at[r, col] = exTxt
                                else:
                                    van = LoadLLM(current_read.at[r, col], exTxt, src_lang, tgt_lang)
                                    exEx = current_read.at[r, col]
                                    current_read.at[r, col] = van
                                    exTxt = van
                restore_identifier_column(originRead, current_read)
                createFolder('./' + languageFull)
                current_read.to_csv('./'+ languageFull +'/' + loadFile + '.csv', mode='w', index=False, encoding='utf-8-sig')

            new_data = pd.DataFrame({'Language': [languageFull], 'File': [loadFile], 'Version': [currentVersion]})
            checkList = pd.concat([checkList, new_data], ignore_index=True)
            checkList.to_csv('./checkList.csv', mode='w', index=False, encoding='utf-8-sig')
            print(languageFull + ' ' + loadFile)

def CsvNRemove(loadFile, languageFull):
    file_path = './'+ languageFull +'/' + loadFile + '.csv'
    with open(file_path, 'r', newline='', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        rows = [
            [cell.replace('\n', ' ').replace('\r', ' ').replace('(남성)', ' ') for cell in row]
            for row in reader
        ]
    with open(file_path, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.writer(outfile)
        writer.writerows(rows)

#불러올 데이터들
loadList = ['AccountBox', 'Etc', 'MatchCategory', 'MatchItem', 'Notice', 'Script', 'ShopItem', 'Tutorial']

#완전히 새로운 데이터로 변경
replaceList = []

#현재 버전
currentVersion = '8.3'

# 언어 정보
readLanDF = pd.read_csv('./LanguageList.csv', encoding = 'utf-8')
originLanguageList = ['en','ko','zh-CN','zh-TW','de','fr','es','it','pt','tr','ru','ja','vi','ms','th','id','jw','bn','hi','ar']
languageList = ['ja', 'zh-CN', 'zh-TW', 'vi', 'de', 'ru', 'es', 'ar', 'it', 'ms', 'th', 'tr', 'fr', 'id', 'jw', 'bn', 'hi', 'pt']

# 번역 실행
for lan in range((len(readLanDF) - len(languageList)), len(readLanDF)):
    src_lang = 'English'
    tgt_lang = readLanDF['Language'][lan]
    Convert(loadList, languageList[lan], tgt_lang, replaceList, src_lang, tgt_lang, currentVersion)

for loadFile in loadList:
    originRead = pd.read_csv('./English/' + loadFile + '.csv', encoding = 'utf-8')
    createFolder('./BeforeEnglish/')
    originRead.to_csv('./BeforeEnglish/' + loadFile + '.csv', mode='w', index=False, encoding='utf-8-sig')

Japanese AccountBox
Japanese Etc
Japanese MatchCategory
Japanese MatchItem
Japanese Notice
Japanese Script
Japanese ShopItem
Japanese Tutorial
ChineseSimplified AccountBox
ChineseSimplified Etc
ChineseSimplified MatchCategory
ChineseSimplified MatchItem
ChineseSimplified Notice
ChineseSimplified Script
ChineseSimplified ShopItem
ChineseSimplified Tutorial
ChineseTraditional AccountBox
ChineseTraditional Etc
ChineseTraditional MatchCategory
ChineseTraditional MatchItem
ChineseTraditional Notice
ChineseTraditional Script
ChineseTraditional ShopItem
ChineseTraditional Tutorial
Vietnamese AccountBox
Vietnamese Etc
Vietnamese MatchCategory
Vietnamese MatchItem
Vietnamese Notice
Vietnamese Script
Vietnamese ShopItem
Vietnamese Tutorial
German AccountBox
German Etc
German MatchCategory
German MatchItem
German Notice
German Script
German ShopItem
German Tutorial
Russian AccountBox
Russian Etc
Russian MatchCategory
Russian MatchItem
Russian Notice
Russian Script
Russian ShopItem
Russian Tutoria

In [10]:
# 문제 번역 검토용 휴리스틱
ASCII_LETTER_PATTERN = re.compile(r'[A-Za-z]')
NON_LATIN_TARGET_LANGUAGES = {
    'Arabic',
    'Bengali',
    'ChineseSimplified',
    'ChineseTraditional',
    'Hindi',
    'Japanese',
    'Russian',
    'Thai',
}
ALLOWED_UNCHANGED_TERMS = {
    'top',
    'mid',
    'jungle',
    'ad',
    'm',
    'w',
    'l',
}
BAD_TRANSLATION_PATTERNS = [
    'translated into',
    'target language',
    'since you did not specify',
    'since the target language was not specified',
    'if you need another language',
    'if you intended a different language',
    'please let me know',
    'i have translated',
    'i translated',
]

def english_ratio(text):
    if not isinstance(text, str) or not text.strip():
        return 1.0

    cleaned_text = PROTECTED_TOKEN_PATTERN.sub('', text)
    alpha_chars = [char for char in cleaned_text if char.isalpha()]
    if not alpha_chars:
        return 0.0

    english_chars = sum(1 for char in alpha_chars if ASCII_LETTER_PATTERN.fullmatch(char))
    return english_chars / len(alpha_chars)

def is_allowed_unchanged_term(text):
    if not isinstance(text, str):
        return False
    return text.strip().lower() in ALLOWED_UNCHANGED_TERMS

def needs_retranslation(src_txt, tgt_txt, target_language):
    src_value = '' if pd.isna(src_txt) else str(src_txt).strip()
    tgt_value = '' if pd.isna(tgt_txt) else str(tgt_txt).strip()

    if not src_value:
        return False
    if not tgt_value:
        return True
    if tgt_value == src_value:
        return not is_allowed_unchanged_term(src_value)

    lower_value = tgt_value.lower()
    if any(pattern in lower_value for pattern in BAD_TRANSLATION_PATTERNS):
        return True

    if target_language in NON_LATIN_TARGET_LANGUAGES:
        src_ratio = english_ratio(src_value)
        tgt_ratio = english_ratio(tgt_value)
        src_has_english = src_ratio >= 0.5

        if not src_has_english and tgt_ratio >= 0.55:
            return True
        if src_has_english and tgt_ratio >= 0.85 and tgt_value != src_value:
            return True

    return False

In [11]:
# 문제 번역만 재번역 (빈칸, 원문 동일, 영어 과다, 설명문 섞임 등)
import shutil
from pathlib import Path

def retranslate_problematic():
    base = Path(r"c:/Users/HOSEO/Documents/GitHub/LOLManager_Transfer")
    load_list = ['AccountBox', 'Etc', 'MatchCategory', 'MatchItem', 'Notice', 'Script', 'ShopItem', 'Tutorial']

    # 스캔할 언어 폴더 (English/BeforeEnglish 등 제외)
    skip_dirs = {'English', 'BeforeEnglish', '.conda', '.ipynb_checkpoints', '__pycache__'}
    lang_dirs = []
    for entry in base.iterdir():
        if entry.is_dir() and entry.name not in skip_dirs and (entry / 'AccountBox.csv').exists():
            lang_dirs.append(entry.name)
    lang_dirs = sorted(lang_dirs)

    # LanguageList.csv가 있다면 폴더명→표시용 언어명 매핑 시도
    lang_name_map = {}
    lang_list_path = base / 'LanguageList.csv'
    if lang_list_path.exists():
        try:
            lang_df = pd.read_csv(lang_list_path, encoding='utf-8')
            for _, row in lang_df.iterrows():
                for key in ['Folder', 'folder', 'Dir', 'dir']:
                    if key in row.index and isinstance(row[key], str):
                        lang_name_map[row[key]] = row.get('Language', row.get('language', row[key]))
        except Exception as e:
            print(f"언어 매핑 로드 오류: {e}")

    total_fixed = 0
    report = []

    for lang in lang_dirs:
        tgt_lang_label = lang_name_map.get(lang, lang)
        for file in load_list:
            eng_path = base / 'English' / f'{file}.csv'
            tgt_path = base / lang / f'{file}.csv'
            if not tgt_path.exists() or not eng_path.exists():
                continue
            try:
                eng_df = pd.read_csv(eng_path, encoding='utf-8')
                tgt_df = pd.read_csv(tgt_path, encoding='utf-8')
            except Exception as e:
                print(f"로드 오류: {lang}/{file}: {e}")
                continue

            min_len = min(len(eng_df), len(tgt_df))
            eng_df = eng_df.iloc[:min_len].reset_index(drop=True)
            tgt_df = tgt_df.iloc[:min_len].reset_index(drop=True)

            cols = [c for c in tgt_df.columns if ('Name' in c) or ('Dec' in c)]
            if not cols:
                continue

            backup_dir = base / 'BeforeFix' / lang
            createFolder(str(backup_dir))
            backup_path = backup_dir / f'{file}.csv'
            if not backup_path.exists():
                shutil.copyfile(tgt_path, backup_path)

            fixed = 0
            for col in cols:
                if col not in eng_df.columns:
                    continue
                eng_col = eng_df[col].fillna('').astype(str)
                tgt_col = tgt_df[col].fillna('').astype(str)
                for idx in range(len(tgt_col)):
                    src_txt = eng_col.iloc[idx].strip()
                    tgt_txt = tgt_col.iloc[idx].strip()
                    if needs_retranslation(src_txt, tgt_txt, tgt_lang_label):
                        new_txt = LoadLLM(src_txt, '', 'English', tgt_lang_label)
                        tgt_df.at[idx, col] = preserve_edge_markers(src_txt, new_txt)
                        fixed += 1

            if fixed > 0:
                restore_identifier_column(eng_df, tgt_df)
                tgt_df.to_csv(tgt_path, index=False, encoding='utf-8-sig')
                total_fixed += fixed
                report.append((lang, file, fixed))

    print(f"총 재번역 개수: {total_fixed}")
    for lang, file, fixed in report:
        print(f"{lang}/{file}: {fixed}개 재번역")

# 실행
retranslate_problematic()

총 재번역 개수: 168
Bengali/Etc: 1개 재번역
ChineseSimplified/Etc: 1개 재번역
French/AccountBox: 3개 재번역
French/Etc: 16개 재번역
French/Script: 7개 재번역
German/AccountBox: 1개 재번역
German/Etc: 20개 재번역
German/Script: 9개 재번역
Indonesian/AccountBox: 1개 재번역
Indonesian/Etc: 16개 재번역
Indonesian/Script: 8개 재번역
Italian/AccountBox: 1개 재번역
Italian/Etc: 15개 재번역
Italian/Script: 5개 재번역
Japanese/Etc: 1개 재번역
Javanese/Etc: 14개 재번역
Javanese/Script: 6개 재번역
Malay/Etc: 11개 재번역
Malay/Script: 6개 재번역
Portuguese/Etc: 8개 재번역
Portuguese/Script: 4개 재번역
Russian/Etc: 1개 재번역
Spanish/Etc: 6개 재번역
Spanish/Script: 2개 재번역
Turkish/Etc: 2개 재번역
Turkish/Script: 2개 재번역
Vietnamese/Etc: 1개 재번역


In [ ]:
# League of Legends 용어 검수 및 재번역 (LLM 이용)
import pandas as pd
from pathlib import Path

def verify_and_improve_lol_terminology():
    """
    영어 파일의 LoL 용어가 제대로 번역되었는지 LLM으로 검수
    """
    base = Path(r"c:/Users/HOSEO/Documents/GitHub/LOLManager_Transfer")
    load_list = ['AccountBox', 'Etc', 'MatchCategory', 'MatchItem', 'Notice', 'Script', 'ShopItem', 'Tutorial']
    
    lol_terminology_guide = """You are an expert in League of Legends terminology and esports.
Review the following English text that was supposedly translated from Korean for a LoL esports fan game.

Check if:
1. LoL-specific terms are correctly used (Champion names, items, abilities, etc.)
2. Esports terminology is accurate (Draft, Ban, Pick, Teamfight, etc.)
3. Game mechanics are correctly expressed
4. The tone is competitive and suitable for esports commentary
5. All placeholders, HTML tags, variables, and special symbols must remain exactly as written

If the text has incorrect or awkward LoL terminology, suggest a better translation.
If the text is good, respond with: [OK]
If it needs improvement, respond with: [IMPROVE] better text here
Do not remove or alter symbols like ?, ??, ◀, ▶, {}, [], [0], [1], or tags like <b>...</b>.

Only respond with [OK] or [IMPROVE] + text."""
    
    total_checked = 0
    total_improved = 0
    improvement_report = []
    
    for file in load_list:
        eng_path = base / 'English' / f'{file}.csv'
        if not eng_path.exists():
            continue
        
        try:
            df = pd.read_csv(eng_path, encoding='utf-8')
        except Exception as e:
            print(f"로드 오류: {file}: {e}")
            continue
        
        cols = [c for c in df.columns if ('Name' in c) or ('Dec' in c)]
        improved_rows = []
        
        for col in cols:
            if col not in df.columns:
                continue
            
            for idx in df.index:
                text = str(df.at[idx, col]).strip()
                if not text or len(text) < 3:
                    continue
                
                total_checked += 1
                
                # LLM으로 LoL 용어 검수
                prompt = f"{lol_terminology_guide}\n\nText to review: \"{text}\""
                
                data = {
                    "model": LLM_MODEL,
                    "messages": [{"role": "user", "content": prompt}]
                }
                
                try:
                    response = requests.post(LLM_API_URL, headers={'Content-Type': 'application/json'}, 
                                            json=data, timeout=60*3)
                    response.raise_for_status()
                    result = response.json()
                    llm_response = result['choices'][0]['message']['content'].strip()
                    
                    if llm_response.startswith('[IMPROVE]'):
                        improved_text = llm_response.replace('[IMPROVE]', '').strip()
                        improved_text = preserve_edge_markers(text, improved_text)
                        row_id = df.at[idx, 'ID'] if 'ID' in df.columns else idx
                        
                        df.at[idx, col] = improved_text
                        total_improved += 1
                        improved_rows.append({
                            'row_id': row_id,
                            'column': col,
                            'before': text[:60],
                            'after': improved_text[:60]
                        })
                
                except Exception as e:
                    print(f"LLM 검수 오류 ({file}/{col}): {e}")
        
        # 개선된 항목이 있으면 저장
        if improved_rows:
            restore_identifier_column(df, df)
            df.to_csv(eng_path, index=False, encoding='utf-8-sig')
            improvement_report.append({
                'file': file,
                'count': len(improved_rows),
                'details': improved_rows[:3]  # 처음 3개만
            })
    
    # 결과 출력
    print("=" * 80)
    print("League of Legends 용어 검수 완료")
    print("=" * 80)
    print(f"총 검수 항목: {total_checked}개")
    print(f"개선된 항목: {total_improved}개")
    print()
    
    if improvement_report:
        for report in improvement_report:
            print(f"\n【{report['file']}】- {report['count']}개 개선")
            for detail in report['details']:
                print(f"  ID {detail['row_id']} ({detail['column']})")
                print(f"    변경 전: \"{detail['before']}...\"")
                print(f"    변경 후: \"{detail['after']}...\"")
            if report['count'] > 3:
                print(f"  ... 외 {report['count']-3}개 항목 개선")
    else:
        print("✓ 모든 League of Legends 용어가 적절하게 번역되었습니다!")
    
    print("\n" + "=" * 80)

print("League of Legends 게임 용어 검수 시작...")
verify_and_improve_lol_terminology()

League of Legends 게임 용어 검수 시작...
League of Legends 용어 검수 완료
총 검수 항목: 716개
개선된 항목: 628개


【AccountBox】- 47개 개선
  ID 1002 (Name)
    변경 전: "Support..."
    변경 후: "Support (role)..."
  ID 1003 (Name)
    변경 전: "Review..."
    변경 후: "Please provide the actual English text you want reviewed. Th..."
  ID 1005 (Name)
    변경 전: "Silver rank achievement..."
    변경 후: "Achieved Silver Rank​..."
  ... 외 44개 항목 개선

【Etc】- 180개 개선
  ID 1000 (Dec)
    변경 전: "Sound effect..."
    변경 후: "The phrase “Sound effect” is too generic and doesn’t convey ..."
  ID 1001 (Dec)
    변경 전: "Background Sound..."
    변경 후: "The phrase “Background Sound” does not correspond to any Lea..."
  ID 1002 (Dec)
    변경 전: "Language..."
    변경 후: "The provided text ("Language") does not contain any LoL or e..."
  ... 외 177개 항목 개선

【MatchCategory】- 58개 개선
  ID 1000000 (Dec)
    변경 전: "Defeat all the KR League teams in the 18 season...."
    변경 후: "Defeat every LCK team in Season 18...."
  ID 1000001 (Dec)
    변경 전: "Defeat all

In [ ]:
# 1단계: 망가진 English 파일 복구 (BeforeEnglish에서)
import pandas as pd
from pathlib import Path
import shutil

def restore_english_files():
    base = Path(r"c:/Users/HOSEO/Documents/GitHub/LOLManager_Transfer")
    load_list = ['AccountBox', 'Etc', 'MatchItem', 'Notice', 'Script', 'ShopItem', 'Tutorial']  # MatchCategory 제외
    
    for file in load_list:
        before_path = base / 'BeforeEnglish' / f'{file}.csv'
        eng_path = base / 'English' / f'{file}.csv'
        
        if before_path.exists():
            shutil.copyfile(before_path, eng_path)
            print(f"복구: {file}.csv")
        else:
            print(f"백업 없음: {file}.csv")
    
    print("\n복구 완료!")

restore_english_files()

In [ ]:
# League of Legends 용어 검수 및 개선 (개선된 버전 - 설명 제거, MatchCategory 포함)
import pandas as pd
from pathlib import Path
import re

def clean_llm_response(text):
    """LLM 응답에서 불필요한 설명, 따옴표, 마크다운 제거"""
    # [OK], [IMPROVE] 태그 제거
    text = re.sub(r'\[OK\]|\[IMPROVE\]', '', text, flags=re.IGNORECASE).strip()
    
    # 설명 문구 패턴 제거
    patterns = [
        r'The sentence does not contain.*?$',
        r'This.*?does not.*?$',
        r'A more suitable.*?:',
        r'Better translation.*?:',
        r'Improved version.*?:',
        r'\*\*.*?\*\*',  # 마크다운 강조
        r'^"(.*)"$',  # 앞뒤 따옴표
        r'→',  # 화살표
    ]
    
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE).strip()
    
    # 여러 줄인 경우 첫 번째 실제 내용만 추출
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if lines:
        # 설명이 아닌 실제 번역문 찾기
        for line in lines:
            if not any(skip in line.lower() for skip in ['league of legends', 'esports', 'better', 'suitable', 'improved']):
                return line.strip('"').strip()
        return lines[0].strip('"').strip()
    
    return text.strip('"').strip()

def verify_and_improve_lol_terminology_v2():
    """
    영어 파일의 LoL 용어 검수 (개선 버전)
    - MatchCategory 포함
    - KR, NA 같은 국가명 유지
    - 불필요한 설명 제거
    """
    base = Path(r"c:/Users/HOSEO/Documents/GitHub/LOLManager_Transfer")
    load_list = ['AccountBox', 'Etc', 'MatchCategory', 'MatchItem', 'Notice', 'Script', 'ShopItem', 'Tutorial']
    
    # 더 간단하고 직접적인 프롬프트
    lol_terminology_guide = """You are a League of Legends esports expert translator.
Review this English text from a LoL esports fan game translation.

IMPORTANT RULES:
- Keep country/region codes like "KR", "NA", "EU" exactly as they are (these refer to countries, not leagues)
- Only suggest changes if LoL/esports terms are CLEARLY wrong
- Respond ONLY with the corrected text, no explanations
- If text is acceptable, respond with just: OK
- Preserve all placeholders, HTML tags, variables, and special symbols exactly as written
- Do not remove or alter symbols like ?, ??, ◀, ▶, {}, [], [0], [1], or tags like <b>...</b>

Text:"""
    
    total_checked = 0
    total_improved = 0
    improvement_report = []
    
    for file in load_list:
        eng_path = base / 'English' / f'{file}.csv'
        if not eng_path.exists():
            continue
        
        print(f"\n검수 중: {file}.csv")
        
        try:
            df = pd.read_csv(eng_path, encoding='utf-8')
        except Exception as e:
            print(f"  로드 오류: {e}")
            continue
        
        cols = [c for c in df.columns if ('Name' in c) or ('Dec' in c)]
        improved_rows = []
        
        for col in cols:
            if col not in df.columns:
                continue
            
            for idx in df.index:
                original_text = str(df.at[idx, col]).strip()
                if not original_text or len(original_text) < 3 or original_text == 'nan':
                    continue
                
                total_checked += 1
                
                # LLM으로 검수
                prompt = f"{lol_terminology_guide} \"{original_text}\""
                
                data = {
                    "model": LLM_MODEL,
                    "messages": [{"role": "user", "content": prompt}]
                }
                
                try:
                    response = requests.post(LLM_API_URL, headers={'Content-Type': 'application/json'}, 
                                            json=data, timeout=60*3)
                    response.raise_for_status()
                    result = response.json()
                    llm_response = result['choices'][0]['message']['content'].strip()
                    
                    # 응답 정리
                    cleaned_response = clean_llm_response(llm_response)
                    cleaned_response = preserve_edge_markers(original_text, cleaned_response)
                    
                    # OK가 아니고, 원본과 다르며, 실제 개선된 내용이 있으면 적용
                    if cleaned_response.upper() != 'OK' and cleaned_response != original_text and len(cleaned_response) > 2:
                        row_id = df.at[idx, 'ID'] if 'ID' in df.columns else idx
                        
                        df.at[idx, col] = cleaned_response
                        total_improved += 1
                        improved_rows.append({
                            'row_id': row_id,
                            'column': col,
                            'before': original_text[:50],
                            'after': cleaned_response[:50]
                        })
                        print(f"  개선: ID {row_id} ({col})")
                
                except Exception as e:
                    print(f"  LLM 검수 오류 (ID {df.at[idx, 'ID'] if 'ID' in df.columns else idx}): {e}")
        
        # 개선된 항목이 있으면 저장
        if improved_rows:
            restore_identifier_column(df, df)
            df.to_csv(eng_path, index=False, encoding='utf-8-sig')
            improvement_report.append({
                'file': file,
                'count': len(improved_rows),
                'details': improved_rows[:5]  # 처음 5개
            })
            print(f"  ✓ {file}.csv 저장 완료 ({len(improved_rows)}개 개선)")
    
    # 결과 출력
    print("\n" + "=" * 80)
    print("League of Legends 용어 검수 완료")
    print("=" * 80)
    print(f"총 검수 항목: {total_checked}개")
    print(f"개선된 항목: {total_improved}개")
    print()
    
    if improvement_report:
        for report in improvement_report:
            print(f"\n【{report['file']}】- {report['count']}개 개선")
            for detail in report['details']:
                print(f"  ID {detail['row_id']} ({detail['column']})")
                print(f"    Before: {detail['before']}...")
                print(f"    After:  {detail['after']}...")
            if report['count'] > 5:
                print(f"  ... 외 {report['count']-5}개")
    else:
        print("✓ 모든 항목이 적절합니다!")
    
    print("\n" + "=" * 80)

print("League of Legends 용어 검수 시작 (MatchCategory 포함, 국가명 유지)...")
verify_and_improve_lol_terminology_v2()

League of Legends 용어 검수 시작 (MatchCategory 포함, 국가명 유지)...

검수 중: AccountBox.csv
  개선: ID 1011 (Name)
  개선: ID 1000 (Dec)
  개선: ID 1002 (Dec)
  개선: ID 1007 (Dec)
  개선: ID 1008 (Dec)
  개선: ID 1010 (Dec)
  개선: ID 1021 (Dec)
  개선: ID 1026 (Dec)
  개선: ID 1027 (Dec)
  개선: ID 1030 (Dec)
  ✓ AccountBox.csv 저장 완료 (10개 개선)

검수 중: Etc.csv
  개선: ID 1003 (Dec)
  개선: ID 1004 (Dec)
  개선: ID 1007 (Dec)
  개선: ID 1008 (Dec)
  개선: ID 1009 (Dec)
  개선: ID 1014 (Dec)
  개선: ID 1015 (Dec)
  개선: ID 1016 (Dec)
  개선: ID 1038 (Dec)
  개선: ID 1045 (Dec)
  개선: ID 1047 (Dec)
  개선: ID 1049 (Dec)
  개선: ID 1051 (Dec)
  개선: ID 1052 (Dec)
  개선: ID 1053 (Dec)
  개선: ID 1054 (Dec)
  개선: ID 1056 (Dec)
  개선: ID 1063 (Dec)
  개선: ID 1065 (Dec)
  개선: ID 1068 (Dec)
  개선: ID 1070 (Dec)
  개선: ID 1091 (Dec)
  개선: ID 1093 (Dec)
  개선: ID 1094 (Dec)
  개선: ID 1103 (Dec)
  개선: ID 1106 (Dec)
  개선: ID 1109 (Dec)
  개선: ID 1114 (Dec)
  개선: ID 1120 (Dec)
  개선: ID 1130 (Dec)
  개선: ID 1132 (Dec)
  개선: ID 1136 (Dec)
  개선: ID 1166 (Dec)
  개선: ID 11